In [ ]:
# =====================================================
# 0~3. 라이브러리 / 파일 설정 / 폰트 / 데이터 불러오기
# =====================================================

from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

OUTPUT_DIR = Path(r"C:\Users\soyou\OneDrive\바탕 화면\textdata\arsenal_pipeline_outputs")
INPUT_FILE = OUTPUT_DIR / "arsenal_youtube_cafe_final_labeled.xlsx"
SAVE_DIR = OUTPUT_DIR / "arsenal_sentiment_charts"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

def set_korean_font():
    font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
    installed_fonts = [f.name for f in fm.fontManager.ttflist]

    for font in font_candidates:
        if font in installed_fonts:
            plt.rcParams["font.family"] = font
            print(f"한글 폰트 설정 완료: {font}")
            break

    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()

MAIN_COLOR = "#132257"
POINT_COLOR = "#38A3FF"
LIGHT_COLOR = "#D9E2EC"
SUB_COLOR = "#52616B"
BG_COLOR = "#F7F9FC"

sentiment_order = ["positive", "neutral", "negative"]

sentiment_kor = {
    "positive": "긍정",
    "neutral": "중립",
    "negative": "부정",
}

sentiment_colors = {
    "positive": "#38A3FF",
    "neutral": "#D9E2EC",
    "negative": "#132257",
}

final_df = pd.read_excel(INPUT_FILE)

plot_df = final_df.copy()

plot_df["gpt_sentiment"] = (
    plot_df["gpt_sentiment"]
    .fillna("neutral")
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "긍정": "positive",
        "중립": "neutral",
        "부정": "negative",
    })
)

plot_df["source"] = (
    plot_df["source"]
    .astype(str)
    .str.lower()
    .str.strip()
)

comment_dt = pd.to_datetime(plot_df["comment_published_at"], errors="coerce")
content_dt = pd.to_datetime(plot_df["content_published_at"], errors="coerce")
plot_df["plot_date"] = comment_dt.fillna(content_dt)
plot_df["plot_date"] = pd.to_datetime(plot_df["plot_date"].dt.date)

print("데이터 크기:", plot_df.shape)
print("\nsource 분포")
print(plot_df["source"].value_counts(dropna=False))
print("\n감정 분포")
print(plot_df["gpt_sentiment"].value_counts(dropna=False))

display(plot_df.head())

In [ ]:
# =====================================================
# 4. YouTube 라운드별 감정 비율 - stacked bar
# =====================================================

youtube_df = plot_df[plot_df["source"] == "youtube"].copy()

def round_sort_key(r):
    m = re.search(r"(\d+)R", str(r))
    if m:
        return (0, int(m.group(1)))
    return (1, str(r))

round_order = sorted(youtube_df["round"].dropna().astype(str).unique(), key=round_sort_key)

youtube_round = pd.crosstab(
    youtube_df["round"],
    youtube_df["gpt_sentiment"],
    normalize="index"
) * 100

youtube_round = (
    youtube_round
    .reindex(index=round_order)
    .reindex(columns=sentiment_order)
    .fillna(0)
)

round_counts = youtube_df["round"].value_counts()

fig, ax = plt.subplots(figsize=(12.5, 6))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

x = np.arange(len(youtube_round))
bottom = np.zeros(len(youtube_round))

for sentiment in sentiment_order:
    values = youtube_round[sentiment].values

    ax.bar(
        x,
        values,
        bottom=bottom,
        color=sentiment_colors[sentiment],
        edgecolor="white",
        linewidth=1.5,
        label=sentiment_kor[sentiment],
    )

    for i, value in enumerate(values):
        if value >= 8:
            ax.text(
                x[i],
                bottom[i] + value / 2,
                f"{value:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                fontweight="bold",
                color="white" if sentiment == "negative" else MAIN_COLOR,
            )

    bottom += values

for i, r in enumerate(youtube_round.index):
    ax.text(
        x[i],
        102,
        f"n={round_counts.get(r, 0):,}",
        ha="center",
        va="bottom",
        fontsize=9,
        color=SUB_COLOR,
    )

ax.set_title("YouTube 댓글 라운드별 감정 비율", fontsize=17, fontweight="bold", color=MAIN_COLOR, pad=20)
ax.text(0.5, 1.03, "아스날 관련 YouTube 댓글 기준", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, color=SUB_COLOR)

ax.set_xlabel("라운드", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_ylabel("감정 비율(%)", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(youtube_round.index, fontsize=11, color=MAIN_COLOR)
ax.set_ylim(0, 110)
ax.tick_params(axis="y", labelsize=10, colors=SUB_COLOR)
ax.grid(axis="y", linestyle="--", alpha=0.25, color="#9AA6B2")

ax.legend(title="감정", loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=10, title_fontsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D7DE")
ax.spines["bottom"].set_color("#D0D7DE")

plt.tight_layout()
plt.savefig(SAVE_DIR / "01_youtube_round_sentiment_bar.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================
# 5. YouTube 라운드별 감정 추세 - line plot
# =====================================================

fig, ax = plt.subplots(figsize=(12.5, 5.8))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

x = np.arange(len(youtube_round))

for sentiment in sentiment_order:
    ax.plot(
        x,
        youtube_round[sentiment].values,
        marker="o",
        linewidth=2.8,
        markersize=6.5,
        color=sentiment_colors[sentiment],
        label=sentiment_kor[sentiment],
    )

    for i, value in enumerate(youtube_round[sentiment].values):
        ax.text(
            x[i],
            value + 2.2,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8.5,
            color=sentiment_colors[sentiment],
            fontweight="bold",
        )

ax.set_title("YouTube 댓글 라운드별 감정 추세", fontsize=17, fontweight="bold", color=MAIN_COLOR, pad=20)
ax.text(0.5, 1.03, "경기 흐름에 따른 긍정·중립·부정 비율 변화", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, color=SUB_COLOR)

ax.set_xlabel("라운드", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_ylabel("감정 비율(%)", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(youtube_round.index, fontsize=11, color=MAIN_COLOR)
ax.set_ylim(0, 100)
ax.tick_params(axis="y", labelsize=10, colors=SUB_COLOR)
ax.grid(axis="y", linestyle="--", alpha=0.25, color="#9AA6B2")

ax.legend(title="감정", loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False, fontsize=10, title_fontsize=10)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D7DE")
ax.spines["bottom"].set_color("#D0D7DE")

plt.tight_layout()
plt.savefig(SAVE_DIR / "02_youtube_round_sentiment_line.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================
# 6. 카페 38R 전후 감정 비율
# =====================================================

cafe_df = plot_df[plot_df["source"] == "cafe"].copy()

period_order = ["before_38R", "after_38R"]
period_label_map = {
    "before_38R": "38R 이전",
    "after_38R": "38R 이후",
}

cafe_period = pd.crosstab(
    cafe_df["period"],
    cafe_df["gpt_sentiment"],
    normalize="index"
) * 100

cafe_period = (
    cafe_period
    .reindex(index=[p for p in period_order if p in cafe_period.index])
    .reindex(columns=sentiment_order)
    .fillna(0)
)

period_counts = cafe_df["period"].value_counts()
period_labels = [period_label_map.get(p, p) for p in cafe_period.index]

fig, ax = plt.subplots(figsize=(9.5, 5))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

left = np.zeros(len(cafe_period))

for sentiment in sentiment_order:
    values = cafe_period[sentiment].values

    bars = ax.barh(
        period_labels,
        values,
        left=left,
        color=sentiment_colors[sentiment],
        edgecolor="white",
        linewidth=1.8,
        label=sentiment_kor[sentiment],
    )

    for bar, value, lft in zip(bars, values, left):
        if value >= 7:
            ax.text(
                lft + value / 2,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%",
                ha="center",
                va="center",
                fontsize=10,
                fontweight="bold",
                color="white" if sentiment == "negative" else MAIN_COLOR,
            )

    left += values

for i, p in enumerate(cafe_period.index):
    ax.text(101.5, i, f"n={period_counts.get(p, 0):,}", va="center", fontsize=10, color=SUB_COLOR)

ax.set_title("네이버 카페 38R 전후 감정 비율 비교", fontsize=17, fontweight="bold", color=MAIN_COLOR, pad=20)
ax.text(0.5, 1.03, "38R 전후 네이버 카페 게시글/댓글의 감정 변화", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, color=SUB_COLOR)

ax.set_xlabel("감정 비율(%)", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_ylabel("시점", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_xlim(0, 110)
ax.grid(axis="x", linestyle="--", alpha=0.25, color="#9AA6B2")

ax.legend(title="감정", loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=3, frameon=False, fontsize=10, title_fontsize=10)

ax.tick_params(axis="x", colors=SUB_COLOR)
ax.tick_params(axis="y", colors=MAIN_COLOR)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D7DE")
ax.spines["bottom"].set_color("#D0D7DE")

plt.tight_layout()
plt.savefig(SAVE_DIR / "03_cafe_before_after_sentiment.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================
# 7. YouTube vs Cafe 전체 감정 비율
# =====================================================

source_sentiment = pd.crosstab(
    plot_df["source"],
    plot_df["gpt_sentiment"],
    normalize="index"
) * 100

source_sentiment = (
    source_sentiment
    .reindex(index=["youtube", "cafe"])
    .reindex(columns=sentiment_order)
    .fillna(0)
)

source_counts = plot_df["source"].value_counts()

source_label_map = {
    "youtube": "YouTube 댓글",
    "cafe": "네이버 카페",
}

source_labels = [source_label_map.get(s, s) for s in source_sentiment.index]

fig, ax = plt.subplots(figsize=(10, 4.8))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

left = np.zeros(len(source_sentiment))

for sentiment in sentiment_order:
    values = source_sentiment[sentiment].values

    bars = ax.barh(
        source_labels,
        values,
        left=left,
        color=sentiment_colors[sentiment],
        edgecolor="white",
        linewidth=1.8,
        label=sentiment_kor[sentiment],
    )

    for bar, value, lft in zip(bars, values, left):
        if value >= 7:
            ax.text(
                lft + value / 2,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%",
                ha="center",
                va="center",
                fontsize=10,
                fontweight="bold",
                color="white" if sentiment == "negative" else MAIN_COLOR,
            )

    left += values

for i, src in enumerate(source_sentiment.index):
    ax.text(101.5, i, f"n={source_counts.get(src, 0):,}", va="center", fontsize=10, color=SUB_COLOR)

ax.set_title("YouTube 댓글 vs 네이버 카페 감정 비율", fontsize=17, fontweight="bold", color=MAIN_COLOR, pad=20)
ax.text(0.5, 1.03, "데이터 출처별 긍정·중립·부정 비율 비교", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, color=SUB_COLOR)

ax.set_xlabel("감정 비율(%)", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_ylabel("데이터 출처", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_xlim(0, 110)
ax.grid(axis="x", linestyle="--", alpha=0.25, color="#9AA6B2")

ax.legend(title="감정", loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=3, frameon=False, fontsize=10, title_fontsize=10)

ax.tick_params(axis="x", colors=SUB_COLOR)
ax.tick_params(axis="y", colors=MAIN_COLOR)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D7DE")
ax.spines["bottom"].set_color("#D0D7DE")

plt.tight_layout()
plt.savefig(SAVE_DIR / "04_source_sentiment_ratio.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================
# 8. YouTube vs Cafe 긍정/부정만 비교
# =====================================================

pn_df = plot_df[plot_df["gpt_sentiment"].isin(["positive", "negative"])].copy()

pn_table = pd.crosstab(
    pn_df["source"],
    pn_df["gpt_sentiment"],
    normalize="index"
) * 100

pn_table = (
    pn_table
    .reindex(index=["youtube", "cafe"])
    .reindex(columns=["positive", "negative"])
    .fillna(0)
)

pn_counts = pn_df["source"].value_counts()

fig, ax = plt.subplots(figsize=(9.5, 4.8))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

left = np.zeros(len(pn_table))

for sentiment in ["positive", "negative"]:
    values = pn_table[sentiment].values

    bars = ax.barh(
        [source_label_map.get(s, s) for s in pn_table.index],
        values,
        left=left,
        color=sentiment_colors[sentiment],
        edgecolor="white",
        linewidth=1.8,
        label=sentiment_kor[sentiment],
    )

    for bar, value, lft in zip(bars, values, left):
        if value >= 5:
            ax.text(
                lft + value / 2,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.1f}%",
                ha="center",
                va="center",
                fontsize=11,
                fontweight="bold",
                color="white" if sentiment == "negative" else MAIN_COLOR,
            )

    left += values

for i, src in enumerate(pn_table.index):
    ax.text(101.5, i, f"n={pn_counts.get(src, 0):,}", va="center", fontsize=10, color=SUB_COLOR)

ax.set_title("YouTube vs 네이버 카페 긍·부정 방향성 비교", fontsize=17, fontweight="bold", color=MAIN_COLOR, pad=20)
ax.text(0.5, 1.03, "중립을 제외하고 positive / negative 비율만 비교", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, color=SUB_COLOR)

ax.set_xlabel("긍·부정 비율(%)", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_ylabel("데이터 출처", fontsize=12, color=MAIN_COLOR, labelpad=10)
ax.set_xlim(0, 110)
ax.grid(axis="x", linestyle="--", alpha=0.25, color="#9AA6B2")

ax.legend(title="감정", loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False, fontsize=10, title_fontsize=10)

ax.tick_params(axis="x", colors=SUB_COLOR)
ax.tick_params(axis="y", colors=MAIN_COLOR)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#D0D7DE")
ax.spines["bottom"].set_color("#D0D7DE")

plt.tight_layout()
plt.savefig(SAVE_DIR / "05_source_positive_negative_ratio.png", dpi=300, bbox_inches="tight")
plt.show()